In [1]:
import os, warnings, csv, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from tqdm import tqdm

In [ ]:
import cv2, os, glob, math, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from tqdm import tqdm

class GenotypeAnalyzer:
    def __init__(self, folder_path, font_scaling=1.0):
        self.folder = folder_path
        self.male_out = os.path.join(folder_path, '_Output_Males')
        self.female_out = os.path.join(folder_path, '_Output_Females')
        self.font_scaling = font_scaling
        os.makedirs(self.male_out, exist_ok=True)
        os.makedirs(self.female_out, exist_ok=True)

    def load_csv_from_subfolders(self):
        keys = ['genotype', 'position', 'lp', 'mp', 'hp']
        filenames = ['genotype_metadata.csv', 'position_Total_df.csv',
                     'percentage_LP_df.csv', 'percentage_MP_df.csv', 'percentage_HP_df.csv']
        data = {k: [] for k in keys}
        for subfolder in os.listdir(self.folder):
            subfolder_path = os.path.join(self.folder, subfolder)
            if not os.path.isdir(subfolder_path):
                continue
            for key, file in zip(keys, filenames):
                file_path = os.path.join(subfolder_path, file)
                if os.path.exists(file_path):
                    data[key].append(pd.read_csv(file_path))
        return data['genotype'], data['position'], data['lp'], data['mp'], data['hp']

    def aggregate_genotype_measurement_data(self, geno_list, pos_list, lp_list, mp_list, hp_list):
        def extract_measurement(df_list, gender, rep_label, vial_identifier):
            ts_dict = {}
            for df in df_list:
                subset = df[df['Vial'] == vial_identifier]
                if not subset.empty:
                    return pd.Series(subset["MEAN"].values, index=subset["Seconds"])
            print(f"Warning: {vial_identifier} not found in measurement data. Skipping replicate {rep_label}.")
            return None

        data = {'position': {"M": {}, "F": {}}, 'lp': {"M": {}, "F": {}},
                'mp': {"M": {}, "F": {}}, 'hp': {"M": {}, "F": {}}}
        counters = {"M": {}, "F": {}}
        for geno_df, pos_df, lp_df, mp_df, hp_df in zip(geno_list, pos_list, lp_list, mp_list, hp_list):
            for _, row in geno_df.iterrows():
                gender = row['Gender']
                genotype = row['Genotype']
                vial_id = f"Vial_{row['Vial_Num']}"
                counters[gender][genotype] = counters[gender].get(genotype, 0) + 1
                rep_label = f"{genotype}_rep{counters[gender][genotype]}"
                for key, df in zip(['position', 'lp', 'mp', 'hp'], [pos_df, lp_df, mp_df, hp_df]):
                    subset = df[df['Vial'] == vial_id]
                    if not subset.empty:
                        ts = pd.Series(subset["MEAN"].values, index=subset["Seconds"])
                        data[key][gender][rep_label] = ts
                    else:
                        print(f"Warning: {vial_id} not found in {key} df. Skipping replicate {rep_label}.")

        # def build_df(ts_dict):
        #     df = pd.DataFrame(ts_dict).T
        #     df = df.reindex(sorted(df.columns, key=lambda x: float(x)), axis=1)
        #     df.reset_index(inplace=True)
        #     df.rename(columns={"index": "Unnamed: 0"}, inplace=True)
        #     df.columns = ["Unnamed: 0"] + [str(col) for col in df.columns if col != "Unnamed: 0"]
        #     return df

        def build_df(ts_dict):
            df = pd.DataFrame(ts_dict).T
            df = df.reindex(sorted(df.columns, key=lambda x: float(x)), axis=1)
            df.reset_index(inplace=True)
            df.rename(columns={"index": "Unnamed: 0"}, inplace=True)
            def sort_key(label):
                genotype, rep_str = label.rsplit('_rep', 1)
                return (genotype, int(rep_str))
            df = df.sort_values(
                by="Unnamed: 0",
                key=lambda col: col.map(sort_key),
                ignore_index=True
            )
            df.columns = ["Unnamed: 0"] + [str(c) for c in df.columns if c != "Unnamed: 0"]
            return df


        def compute_velocity(pos_df):
            time_strs = [col for col in pos_df.columns if col != "Unnamed: 0"]
            time_points = np.array([float(t) for t in time_strs])
            rows = []
            for _, row in pos_df.iterrows():
                rep_label = row["Unnamed: 0"]
                pos = row[time_strs].astype(float).values
                vel = np.gradient(pos, time_points)
                rows.append([rep_label] + vel.tolist())
            return pd.DataFrame(rows, columns=pos_df.columns)

        male_pos, female_pos = build_df(data['position']['M']), build_df(data['position']['F'])
        male_lp, female_lp = build_df(data['lp']['M']), build_df(data['lp']['F'])
        male_mp, female_mp = build_df(data['mp']['M']), build_df(data['mp']['F'])
        male_hp, female_hp = build_df(data['hp']['M']), build_df(data['hp']['F'])
        male_vel, female_vel = compute_velocity(male_pos), compute_velocity(female_pos)

        return (male_pos, female_pos, male_vel, female_vel,
                male_lp, female_lp, male_mp, female_mp, male_hp, female_hp)

    def create_plot_aggregated_df(self, df, gender, output_folder, measurement):
        time_cols = df.columns[1:]
        grouped = {}
        for _, row in df.iterrows():
            genotype = row["Unnamed: 0"].rsplit("_rep", 1)[0]
            grouped.setdefault(genotype, []).append(row[time_cols].astype(float))
    
        agg_df = pd.DataFrame(index=[float(t) for t in time_cols])
        for genotype, reps in grouped.items():
            reps_df = pd.DataFrame(reps)
            mean, sem, count = reps_df.mean(), reps_df.std(ddof=1) / np.sqrt(len(reps)), reps_df.count()
            agg_df[f"{genotype}_mean"] = mean.values
            agg_df[f"{genotype}_sem"] = sem.values
            agg_df[f"{genotype}_N"] = count.values
    
        agg_df.index.name = "Time (seconds)"
        agg_df.to_csv(os.path.join(output_folder, f"{gender.lower()}_agg_{measurement}_data.csv"), index=False)
    
        plt.figure(figsize=(10, 6))
        for col in [c for c in agg_df.columns if c.endswith("_mean")]:
            genotype = col[:-5]
            plt.errorbar(
                agg_df.index,
                agg_df[col],
                yerr=np.nan_to_num(agg_df[f"{genotype}_sem"]),
                capsize=3,
                label=f"{genotype} (N={int(agg_df[f'{genotype}_N'].iloc[0])})",
                marker='o',
                linestyle='-'
            )
    
        if measurement == "velocity":
            plt.axhline(0, color='black', linestyle='--')
    
        plt.xlabel("Time (seconds)", fontsize=12 * self.font_scaling)
        ylabel = {
            "velocity": "Velocity (cm/sec)",
            "position": "Position (cm)"
        }.get(measurement, f"{measurement.capitalize()} %")
        plt.ylabel(ylabel, fontsize=12 * self.font_scaling)
        plt.title(f"{gender.capitalize()} Genotype Aggregated {measurement.capitalize()} Over Time", fontsize=14 * self.font_scaling)
        plt.legend(fontsize=10 * self.font_scaling)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f"{gender.lower()}_plot_{measurement}.tiff"), dpi=300, format='tiff')
        plt.show()
        plt.close()

    def run(self):
        geno_list, pos_list, lp_list, mp_list, hp_list = self.load_csv_from_subfolders()
        results = self.aggregate_genotype_measurement_data(geno_list, pos_list, lp_list, mp_list, hp_list)
    
        genders = ['male', 'female']
        measurements = ['position', 'velocity', 'low-performer', 'middle-performer', 'high-performer']
        output_folders = [self.male_out, self.female_out]
    
        data_mapping = {
            'position': results[0:2],
            'velocity': results[2:4],
            'low-performer': results[4:6],
            'middle-performer': results[6:8],
            'high-performer': results[8:10]
        }
    
        filename_suffix = {
            'position': 'pos_data.csv',
            'velocity': 'vel_data.csv',
            'low-performer': 'lp_perc_data.csv',
            'middle-performer': 'mp_perc_data.csv',
            'high-performer': 'hp_perc_data.csv'
        }
    
        for i, gender in enumerate(genders):
            for measurement in measurements:
                df = data_mapping[measurement][i]
                out_dir = output_folders[i]
                df.to_csv(os.path.join(out_dir, f"{gender}_stats_{filename_suffix[measurement]}"), index=False)
                self.create_plot_aggregated_df(df, gender, out_dir, measurement)

In [1]:
import warnings
# from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", message="Pandas requires version '1.3.6' or newer of 'bottleneck'")
warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")
warnings.filterwarnings( "ignore", category=FutureWarning, message=".*default of observed=False is deprecated and will be changed to True in a future version of pandas.*" )
# warnings.filterwarnings("ignore", category=PerformanceWarning, message=".*DataFrame is highly fragmented.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, zipfile, cv2,torch, torchvision, subprocess
from tqdm import tqdm
import seaborn as sns
import matplotlib.cm as cm
from PIL import Image
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from scipy.signal import find_peaks

from video_converter import VideoConverter
from video_processor import VideoProcessor
from vial_network import Vial_Network
from gt_process import Finalized_Geotaxis
from gt_data_agg import GenotypeAnalyzer
from statistical_analysis import Statistical_Analysis

In [2]:
def main():
    experiment = input("Enter experiment name (e.g., 'CS_SK2' or 'W1118_CLKOut'): ")
    lamp_var = input("Are you using the lamp? (Yes or No): ").strip().capitalize()
    # frame_steps = int(input("Enter frame steps (e.g., 30 -> every 0.5 secs, 60 -> every 1 secs): "))
    frame_steps = 30
    stats_time_cut = float(input("Enter Stats Time Cut (e.g., 9 -> stats from 0 to 9 seconds): "))
    
    vid_clips = 4  
    temp = list(range(1, vid_clips + 1))

    spec_video_folders = [d for d in os.listdir(f"./{experiment}") 
                         if os.path.isdir(os.path.join(f"./{experiment}", d)) 
                         and d not in ('.ipynb_checkpoints', '_Output_Males', '_Output_Females')]

    for i, spec_video in tqdm(enumerate(spec_video_folders, start=1)):
        if check_required_files(experiment, spec_video):
            print(f"All required files found for {spec_video}. Skipping processing...")
            continue  # Skip to the next folder

        print(f"Processing video folder: {spec_video}")
        # ______________H264_TO_MP4_________________
        input_file = f"./{experiment}/{spec_video}/{spec_video}.h264"
        output_file = f"./{experiment}/{spec_video}/{spec_video}.mp4"

        converter = VideoConverter(input_file, output_file)
        converter.convert()
        del input_file, output_file

        # _____________VIDEO_SNIPS__________________
        video_path = f"./{experiment}/{spec_video}/{spec_video}.mp4"
        print(f"Start Video TRIMS for {spec_video}:")
        processor = VideoProcessor(video_path)
        processor.process_video()
        processor.video_filt()

        trims_ttl = vid_clips 
        for trim_cnt in range(1, trims_ttl + 1):
            start_frame = processor.frame_ranges_df.iloc[trim_cnt - 1]['start_frame']
            end_frame = processor.frame_ranges_df.iloc[trim_cnt - 1]['end_frame']
            processor.crop_video(start_frame, end_frame, trim_cnt)

        video_inputs = [f"./{experiment}/{spec_video}/TRIM_{i}_{spec_video}.mp4" for i in temp]
        output_files = [f"./{experiment}/{spec_video}/{spec_video}_output/TRIM_{i}_{spec_video}_y_positions.csv" for i in temp]
        genotype_csv_pth = f"./{experiment}/{spec_video}/genotype_metadata.csv"
        vials_to_drop, vial_num_list = geno_meta(genotype_csv_pth)

        # ______________VIAL_NETWORK___________________
        vial_pos_lists = []
        print(f"VIALS USED: \n{vial_num_list}\n VIALS USED LENGTH: {len(vial_num_list)}\n")
        for idx, vid in enumerate(video_inputs, start=1):
            print(f"Start Vial Network for {spec_video} TRIM {idx}:")
            vial_network = Vial_Network(experiment, spec_video, idx, vid, vials_to_drop, lamp_var)
            vial_network.predict_and_display()
            # vial_network.save_model("gt_newVial_nn.pth")

            vials_input = f"./{experiment}/{spec_video}/trim_{idx}_{spec_video}_vials_pos.csv"
            vial_pos_lists.append(vials_input)

        #________________GEOTAXIS_MAIN________________#
        print(f"\n\nRUNNING Video {i}/{len(spec_video_folders)}: '{os.path.basename(spec_video)}':\n")
        fin_geo = Finalized_Geotaxis(experiment=experiment, spec_vid=os.path.basename(spec_video), 
                                     fps=60, frame_step=frame_steps, top_thresh=0.50, bottom_thresh=0.55, 
                                     adder_val=150, remove_px=125)
        fin_geo.run()
    
    #________________EXPERIMENT_AGGREGATOR:________________#
    folder_path = f"./{experiment}/"
    analyzer = GenotypeAnalyzer(folder_path)
    analyzer.run()

    #________________STATISTICAL_ANALYSIS________________#
    sa = Statistical_Analysis(experiment, filter_time=stats_time_cut)
    sa.run_analysis()
    
    #________________ZIPPING_FOLDER________________
    input_folder = f'./{experiment}/'
    output_zip_path = f'./{experiment}_ZIPPED.zip'
    zipping = input("Do you want to ZIP the experiment folder? (Yes or No): ").lower()
    should_zip = zipping in ("yes", "y")
    if should_zip:
        zip_folder(input_folder, output_zip_path)

        
def geno_meta(genotype_csv_input, n=12):
    geno_df = pd.read_csv(genotype_csv_input)
    vial_num_list = geno_df["Vial_Num"].tolist()
    irrel_vials = list(set(range(1, n + 1)) - set(geno_df["Vial_Num"].tolist()))
    return irrel_vials, vial_num_list

    
def zip_folder(input_folder, output_zip_path):
    if not os.path.exists(input_folder):
        print(f"Input folder '{input_folder}' does not exist.")
        return
    
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        total_files = sum(len(files) for _, _, files in os.walk(input_folder))
        with tqdm(total=total_files, unit='file', desc='Zipping') as pbar:
            for root, dirs, files in os.walk(input_folder):
                for file in files:
                    file_path = os.path.join(root, file)
                    zipf.write(file_path, os.path.relpath(file_path, input_folder))
                    pbar.update(1)
    
    print(f"Folder '{input_folder}' has been zipped to '{output_zip_path}'.")

def check_required_files(experiment, video_folder):
    required_files_main = [
        f"{video_folder}.h264", 
        f"{video_folder}.mp4", 
        "genotype_metadata.csv",
    ]
    
    required_trim_mp4_files = [
        f"TRIM_1_{video_folder}.mp4", 
        f"TRIM_2_{video_folder}.mp4", 
        f"TRIM_3_{video_folder}.mp4", 
        f"TRIM_4_{video_folder}.mp4"
    ]

    required_trim_csv_files = [
        f"trim_1_{video_folder}_vials_pos.csv", 
        f"trim_2_{video_folder}_vials_pos.csv", 
        f"trim_3_{video_folder}_vials_pos.csv", 
        f"trim_4_{video_folder}_vials_pos.csv"
    ]

    required_csv_outputs = [
        "percentage_HP_df.csv",
        "percentage_LP_df.csv",
        "percentage_MP_df.csv",
        "position_Total_df.csv"
    ]

    required_png_outputs = [
        "percentage_HP_plot.png",
        "percentage_LP_plot.png",
        "percentage_MP_plot.png",
        "position_Total_plot.png"
    ]

    main_folder_path = f"./{experiment}/{video_folder}"
    for file in required_files_main:
        if not os.path.exists(os.path.join(main_folder_path, file)):
            print(f"Missing required main file: {file}")
            return False
    
    for file in required_trim_mp4_files:
        if not os.path.exists(os.path.join(main_folder_path, file)):
            print(f"Missing trimmed video file: {file}")
            return False
            
    for file in required_trim_csv_files:
        if not os.path.exists(os.path.join(main_folder_path, file)):
            print(f"Missing trimmed video file: {file}")
            return False

    for file in required_csv_outputs:
        if not os.path.exists(os.path.join(main_folder_path, file)):
            print(f"Missing trimmed video file: {file}")
            return False
        
    for file in required_png_outputs:
        if not os.path.exists(os.path.join(main_folder_path, file)):
            print(f"Missing trimmed video file: {file}")
            return False
    return True

In [3]:
main()

Enter experiment name (e.g., 'CS_SK2' or 'W1118_CLKOut'):  GLaz_APP
Are you using the lamp? (Yes or No):  yes
Enter Stats Time Cut (e.g., 9 -> stats from 0 to 9 seconds):  10


0it [00:00, ?it/s]


Missing required main file: Archana_SetSet-1_Apr-06_04-07.mp4
Processing video folder: Archana_SetSet-1_Apr-06_04-07


FileNotFoundError: [Errno 2] No such file or directory: 'ffmpeg'